In [0]:
dbutils.widgets.text("table_name", "", "table_name")
dbutils.widgets.text("email_recipient", "", "email_recipient")
dbutils.widgets.text("email_sender", "no-reply@databricks.com", "email_sender")
dbutils.widgets.dropdown("status", "1", ["0", "1"], "status")

table_name = dbutils.widgets.get("table_name")
email_recipient = dbutils.widgets.get("email_recipient").split(',')
email_sender = dbutils.widgets.get("email_sender")
status = dbutils.widgets.get("status")

print("table_name == ",table_name)
print("email_recipient == ",email_recipient)
print("email_sender == ",email_sender)
print("status == ",status)

In [0]:
import configparser
import json

import smtplib
from email.mime.text import MIMEText

from dbruntime.databricks_repl_context import get_context

In [0]:
config = configparser.ConfigParser()
config.read('setup.config')

In [0]:
def get_notebook_run_links(success=True):
    try:
        try:
            ctx = get_context()
            
            # Extract basic host metadata directly
            browser_host = getattr(ctx, "browserHostName", None)
            workspace_id = getattr(ctx, "workspaceId", "N/A")
            
            # Read tags dictionary
            tags = getattr(ctx, "isInJob", {}) # falls back gracefully
            if hasattr(ctx, "get_tags"):
                tags = ctx.get_tags()
            elif hasattr(ctx, "__dict__"):
                tags = ctx.__dict__
        
        # Fallback to safe internal API if REPL context import isn't present
        except ImportError:
            context_str = dbutils.notebook.entry_point.getDbutils().notebook().getContext().safeToJson()
            context = json.loads(context_str)
            tags = context.get("attributes", {}) or context.get("tags", {})
            browser_host = tags.get("browserHostName")
            workspace_id = tags.get("orgId", "N/A")

        # 1. Base URL formulation
        if browser_host:
            base_url = f"https://{browser_host}"
        else:
            try:
                # Direct lookup via Spark config
                base_url = spark.conf.get("spark.databricks.workspaceUrl")
                if not base_url.startswith("http"):
                    base_url = f"https://{base_url}"
                browser_host = base_url.replace("https://", "")
            except Exception:
                workspace_url = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
                base_url = workspace_url.replace("api/2.0", "").rstrip("/")
                browser_host = base_url.replace("https://", "")

        # 2. Extract job-specific parameters
        job_name = tags.get("jobName", "Interactive Notebook")
        job_id = tags.get("jobId")
        
        run_id_obj = tags.get("currentRunId", {})
        run_id = run_id_obj.get("id") if isinstance(run_id_obj, dict) else tags.get("runId")
        job_run_id = tags.get("jobRunId", run_id)
        
        user = tags.get("user", "System")
        launched_by = "Manually" if user != "System" else "Scheduled Workflow"
        
        # 3. Build Safe HTML Link Anchors
        workspace_link = f"<a href='{base_url}/?o={workspace_id}' style='color: #1a73e8; text-decoration: none;'>{browser_host} [{workspace_id}]</a>"
        
        if job_id and job_run_id:
            job_link = f"<a href='{base_url}/#job/{job_id}/run/{job_run_id}' style='color: #1a73e8; text-decoration: none;'>{job_name} [{job_id}]</a>"
        elif job_id:
            job_link = f"<a href='{base_url}/#job/{job_id}' style='color: #1a73e8; text-decoration: none;'>{job_name} [{job_id}]</a>"
        else:
            job_link = "Interactive Notebook (Not a Job Run)"

        return {
            "Workspace": workspace_link,
            "Job": job_link,
            "Job Run": str(job_run_id) if job_run_id else "N/A",
            "Status": "Succeeded" if success else "Failed"
        }
    except Exception as e:
        print(f"Error fetching notebook run details: {e}")
        return {
            "Workspace": "Local Development (No Link)",
            "Job": "Manual Run (No Link)",
            "Job Run": "Local Testing",
            "Status": "Succeeded"
        }

In [0]:
def send_completion_email(table_name, sender_email, recipient_email, config, run_details, success=True):
    # --- Configuration ---
    app_password = config.get('SMTP', 'password')
    smtp_server = config.get('SMTP', 'smtp_server')
    smtp_port = config.getint('SMTP', 'smtp_port')

    # --- Determine Status and Colors ---
    if success:
        status_text = "SUCCESS"
        status_color = "#1a73e8"
        message = "A run of this job has completed successfully"
    else:
        status_text = "FAILURE"
        status_color = "#d93025"
        message = "A run of this job has failed"

    subject = f"DQX Check {status_text.capitalize()} for Table | {table_name}"

    # --- Dynamic Rows Generation for the Box ---
    table_rows = ""
    for key, value in run_details.items():
        table_rows += f"""
        <tr>
            <td style='padding: 8px 0; font-weight: bold; color: #5f6368; width: 30%;'>{key}</td>
            <td style='padding: 8px 0; color: #202124;'>{value}</td>
        </tr>
        """

    # --- HTML Body Construction ---
    body = f"""
    <html>
    <body style="font-family: Arial, sans-serif; line-height: 1.6; color: #333;">
        
        <!-- Header Section -->
        <h2 style="color: {status_color}; font-size: 24px; margin-bottom: 5px;">
            {message}
        </h2>
        <h3 style="font-size: 20px; color: #202124; margin-top: 0; margin-bottom: 20px;">
            Run details:
        </h3>
        
        <!-- Square Box Container -->
        <div style="border: 2px solid #dadce0; border-radius: 8px; padding: 20px; max-width: 600px; background-color: #f8f9fa;">
            <table style="width: 100%; border-collapse: collapse; font-size: 14px;">
                <tbody>
                    <tr>
                        <td style='padding: 8px 0; font-weight: bold; color: #5f6368; width: 30%;'>Table Target</td>
                        <td style='padding: 8px 0; color: #202124; font-weight: bold;'>{table_name}</td>
                    </tr>
                    {table_rows}
                </tbody>
            </table>
        </div>
        
    </body>
    </html>
    """

    msg = MIMEText(body, 'html')
    msg['Subject'] = subject
    msg['From'] = sender_email
    msg['To'] = ','.join(recipient_email)

    try:
        with smtplib.SMTP(smtp_server, smtp_port) as server:
            server.starttls()
            server.login(sender_email, app_password)
            server.sendmail(sender_email, recipient_email, msg.as_string())
        print("Email sent successfully!")
    except Exception as e:
        print(f"Error: {e}")


In [0]:
run_details_dict = get_notebook_run_links(success=(status == "1"))

In [0]:
send_completion_email(
    table_name, 
    email_sender , 
    email_recipient, 
    config ,
    run_details=run_details_dict,
    success=(status == "1")
)